## Actividad 3_6

<div style="border-style:groove;border-width:thin;padding:10px">

En esta actividad vamos a intentar solucionar un problema de regresión con uno de los métodos que hemos visto en clase hasta ahora:
- Regresión Lineal Simple.
- Regresión Lineal Múltiple.
- Regresión Polinómica.
</div>

<p style="border-style:groove;border-width:thin;padding:10px">
Lo primero que vamos a hacer es importar los datos y analizar el dataset que tenemos.
</p>

In [48]:
#Importamos los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

cochesdf = pd.read_csv("CarPrice_Assignment.csv")
cochesdf.corr(numeric_only=True)['price'].abs().sort_values(ascending=False)[1:]

enginesize          0.874145
curbweight          0.835305
horsepower          0.808139
carwidth            0.759325
highwaympg          0.697599
citympg             0.685751
carlength           0.682920
wheelbase           0.577816
boreratio           0.553173
carheight           0.119336
car_ID              0.109093
peakrpm             0.085267
symboling           0.079978
stroke              0.079443
compressionratio    0.067984
Name: price, dtype: float64

In [49]:
#Pintamos las relaciones entre las variables.
#sns.pairplot(cochesdf)

In [50]:
#Vemos la pinta de los datos.
print(cochesdf.head())

   car_ID  symboling                   CarName fueltype aspiration doornumber  \
0       1          3        alfa-romero giulia      gas        std        two   
1       2          3       alfa-romero stelvio      gas        std        two   
2       3          1  alfa-romero Quadrifoglio      gas        std        two   
3       4          2               audi 100 ls      gas        std       four   
4       5          2                audi 100ls      gas        std       four   

       carbody drivewheel enginelocation  wheelbase  ...  enginesize  \
0  convertible        rwd          front       88.6  ...         130   
1  convertible        rwd          front       88.6  ...         130   
2    hatchback        rwd          front       94.5  ...         152   
3        sedan        fwd          front       99.8  ...         109   
4        sedan        4wd          front       99.4  ...         136   

   fuelsystem  boreratio  stroke compressionratio horsepower  peakrpm citympg  \

<p style="border-style:groove;border-width:thin;padding:10px">
A continuación vamos a modificar el dataset para eliminar lo que no nos interesa y cambiar las columnas para poder hacer una regresión.
    
</p>

In [51]:
#Quitamos las columnas que no nos interesan.
cochesdf.drop(['CarName','car_ID'],axis=1,inplace=True)
print(cochesdf.head())

   symboling fueltype aspiration doornumber      carbody drivewheel  \
0          3      gas        std        two  convertible        rwd   
1          3      gas        std        two  convertible        rwd   
2          1      gas        std        two    hatchback        rwd   
3          2      gas        std       four        sedan        fwd   
4          2      gas        std       four        sedan        4wd   

  enginelocation  wheelbase  carlength  carwidth  ...  enginesize  fuelsystem  \
0          front       88.6      168.8      64.1  ...         130        mpfi   
1          front       88.6      168.8      64.1  ...         130        mpfi   
2          front       94.5      171.2      65.5  ...         152        mpfi   
3          front       99.8      176.6      66.2  ...         109        mpfi   
4          front       99.4      176.6      66.4  ...         136        mpfi   

  boreratio stroke  compressionratio horsepower  peakrpm  citympg  highwaympg  \
0    

<div style="border-style:groove;border-width:thin;padding:10px">
    Ahora comprobamos si hay valores nulos. A continuación vamos a proceder a cambiar las columnas que tienen categorías ("categorical features") para poder realizar una regresión con ellas. 
<p>Antes que nada, vamos a definir que son columnas categóricas. Son columnas cuyos datos deben pertenecer a un conjunto de valores finito. Este conjunto de valores puede ser numérico (en cuyo caso podemos usarlo directamente en una regresión) o un texto.</p>
<p>Si nos encontramos con columnas con texto, como es nuestro caso, lo más común es asignar valores numéricos a los valores de las columnas. Esta técnica se llama <b>One-hot encoding</b>. El cambio más habitual para poder realizar una regresión sería convertir la columna en varias, una por cada posible valor. Usando como ejemplo nuestro dataset, la columna <b>fueltype</b> se transformaría en 2 columnas, <b>fueltype-gas y fueltype-diesel</b>.</p>
    <p>Los posibles valores de estas columnas dependeran de la codificación que usemos:</p>
    <ul>
        <li><b>Dummy encoding:</b> Tendrán 0 o 1. En nuestro caso de ejemplo, un coche diesel tendrá 0 en fueltype-gas y 1 en fueltype-diesel.</li>
        <li><b>Simple effect encoding:</b> En vez de 0 y 1 tendrán -0,25 y 0,75. En el mismo ejemplo, el coche diesel tendría -0,25 en fueltype-gas y 0,75 en fueltype-diesel.</li>
    </ul>    
    <p>La función <b>get_dummies</b> de pandas nos permite hacer este cambio en una columa o una lista de columnas. Vamos a hacer un ejemplo con una columna y, después, a moficiar las demás.</p>
</div>

In [52]:
#Buscamos valores nulos.
print(cochesdf.isnull().values.any())

False


In [53]:
#Otra manera
cochesdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   symboling         205 non-null    int64  
 1   fueltype          205 non-null    object 
 2   aspiration        205 non-null    object 
 3   doornumber        205 non-null    object 
 4   carbody           205 non-null    object 
 5   drivewheel        205 non-null    object 
 6   enginelocation    205 non-null    object 
 7   wheelbase         205 non-null    float64
 8   carlength         205 non-null    float64
 9   carwidth          205 non-null    float64
 10  carheight         205 non-null    float64
 11  curbweight        205 non-null    int64  
 12  enginetype        205 non-null    object 
 13  cylindernumber    205 non-null    object 
 14  enginesize        205 non-null    int64  
 15  fuelsystem        205 non-null    object 
 16  boreratio         205 non-null    float64
 1

<div style="border-style:groove;border-width:thin;padding:10px">
    Hay una columna categórica que tiene valores numéricos codificados en texto. En este caso he optado por modificarla y pasarla a un tipo de dato numérico aunque se podría hacer lo mismo que con las demás.
</div>

In [54]:
cochesdf['doornumber'] = cochesdf['doornumber'].replace({'two':2,'four':4})

C:\Users\Equipo\AppData\Local\Temp\ipykernel_18668\2190523151.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cochesdf['doornumber'] = cochesdf['doornumber'].replace({'two':2,'four':4})


In [55]:
#Tenemos una "categorical feature" que es distinta. Tiene datos numéricos codificados con letras. Lo convertimos 
# en números. Sería algo parecido al Label Encoding
print (cochesdf['doornumber'].unique())
puertas = []
for index,row in cochesdf.iterrows():
    if row['doornumber']=='two':
        puertas.append(2)
    else:
        puertas.append(4)
cochesdf['doornumber']=puertas
print(cochesdf.head())

[2 4]
   symboling fueltype aspiration  doornumber      carbody drivewheel  \
0          3      gas        std           4  convertible        rwd   
1          3      gas        std           4  convertible        rwd   
2          1      gas        std           4    hatchback        rwd   
3          2      gas        std           4        sedan        fwd   
4          2      gas        std           4        sedan        4wd   

  enginelocation  wheelbase  carlength  carwidth  ...  enginesize  fuelsystem  \
0          front       88.6      168.8      64.1  ...         130        mpfi   
1          front       88.6      168.8      64.1  ...         130        mpfi   
2          front       94.5      171.2      65.5  ...         152        mpfi   
3          front       99.8      176.6      66.2  ...         109        mpfi   
4          front       99.4      176.6      66.4  ...         136        mpfi   

  boreratio stroke  compressionratio horsepower  peakrpm  citympg  highway

In [56]:
cochesdf.head()

,symboling,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,carlength,carwidth,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,3,gas,std,4,convertible,rwd,front,88.6,168.8,64.1,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,3,gas,std,4,convertible,rwd,front,88.6,168.8,64.1,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,1,gas,std,4,hatchback,rwd,front,94.5,171.2,65.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,2,gas,std,4,sedan,fwd,front,99.8,176.6,66.2,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,2,gas,std,4,sedan,4wd,front,99.4,176.6,66.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [57]:
cochesdf = pd.get_dummies(cochesdf,dtype=int)

In [58]:
cochesdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 205 entries, 0 to 204
Data columns (total 52 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   symboling              205 non-null    int64  
 1   doornumber             205 non-null    int64  
 2   wheelbase              205 non-null    float64
 3   carlength              205 non-null    float64
 4   carwidth               205 non-null    float64
 5   carheight              205 non-null    float64
 6   curbweight             205 non-null    int64  
 7   enginesize             205 non-null    int64  
 8   boreratio              205 non-null    float64
 9   stroke                 205 non-null    float64
 10  compressionratio       205 non-null    float64
 11  horsepower             205 non-null    int64  
 12  peakrpm                205 non-null    int64  
 13  citympg                205 non-null    int64  
 14  highwaympg             205 non-null    int64  
 15  price 

In [59]:
cochesdf.corr(numeric_only=True)['price'].abs().sort_values(ascending=False)[1:]

enginesize               0.874145
curbweight               0.835305
horsepower               0.808139
carwidth                 0.759325
cylindernumber_four      0.697762
highwaympg               0.697599
citympg                  0.685751
carlength                0.682920
drivewheel_rwd           0.638957
drivewheel_fwd           0.601950
wheelbase                0.577816
boreratio                0.553173
fuelsystem_mpfi          0.517075
fuelsystem_2bbl          0.501374
cylindernumber_eight     0.478614
cylindernumber_six       0.474978
enginetype_ohcv          0.385991
enginetype_ohc           0.344270
enginelocation_front     0.324973
enginelocation_rear      0.324973
carbody_hatchback        0.262039
cylindernumber_five      0.249606
carbody_hardtop          0.225854
cylindernumber_twelve    0.199634
carbody_convertible      0.187681
aspiration_std           0.177926
aspiration_turbo         0.177926
fuelsystem_1bbl          0.170945
enginetype_dohcv         0.159225
enginetype_doh

<div style="border-style:groove;border-width:thin;padding:10px">
Para generar los conjuntos X e y vamos a eliminar price en X para coger solo esa columna en y.
</div>

In [60]:
#Generamos los conjuntos X e y.
X = cochesdf.drop(['price'],axis=1)
y = cochesdf['price'].to_frame()
print(X)

     symboling  doornumber  wheelbase  carlength  carwidth  carheight  \
0            3           4       88.6      168.8      64.1       48.8   
1            3           4       88.6      168.8      64.1       48.8   
2            1           4       94.5      171.2      65.5       52.4   
3            2           4       99.8      176.6      66.2       54.3   
4            2           4       99.4      176.6      66.4       54.3   
..         ...         ...        ...        ...       ...        ...   
200         -1           4      109.1      188.8      68.9       55.5   
201         -1           4      109.1      188.8      68.8       55.5   
202         -1           4      109.1      188.8      68.9       55.5   
203         -1           4      109.1      188.8      68.9       55.5   
204         -1           4      109.1      188.8      68.9       55.5   

     curbweight  enginesize  boreratio  stroke  ...  cylindernumber_twelve  \
0          2548         130       3.47    2.6

<div style="border-style:groove;border-width:thin;padding:10px">
Ahora vamos a entrenar el sistema usando un modelo de regresión lineal. ¿Será suficiente? Vamos a usar todas las columnas. También se podría probar a usar un subconjunto de columnas.
</div>

In [61]:
#Dividimos el dataset en los conjuntos de entrenamiento.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [67]:
#Solucionamos el problema con un regresor lineal y analizamos el resultado.
from sklearn import metrics
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)

print('Raiz del error absoluto medio: ', MAE(y_test,y_pred))
from sklearn.metrics import r2_score
print('R cuadrado: ', r2_score(y_test, y_pred))


Raiz del error absoluto medio:  2110.031152607584
R cuadrado:  0.8669546796177385


<div style="border-style:groove;border-width:thin;padding:10px">
El resultado obtenido es 0.87 de R². Está bastante bien. El error cuadrático medio que estamos teniendo en el conjunto de test es de 2100€. Teniendo en cuenta el precio de un coche no parece un error pequeño. Vamos a intentar hacerlo mejor. 
    <p>Si habéis pintado las relaciones entre las distintas columnas habréis visto que hay algunas que parecen tener una relación polinómica con el precio. En concreto de grado 2. Vamos a probar con una regresión polinómica, de nuevo, con todas las columnas.</p>
</div>

In [69]:
#Vamos a probar con una regresión polinómica:
from sklearn.preprocessing import PolynomialFeatures
poly_features = PolynomialFeatures(degree=2,include_bias=False)
X_poly = poly_features.fit_transform(X)
y = cochesdf['price']


X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size = 0.2, random_state = 0)

from sklearn import metrics
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)

print('Raiz del error cuadrático medio: ', MAE(y_test,y_pred))
from sklearn.metrics import r2_score
print('R cuadrado: ', r2_score(y_test, y_pred))
print('R cuadrado entrenamiento: ', r2_score(y_train, lm.predict(X_train)))

Raiz del error cuadrático medio:  115744.0682153313
R cuadrado:  -1174.4329580208482
R cuadrado entrenamiento:  0.9984642634434308


<div style="border-style:groove;border-width:thin;padding:10px">
¿Porqué sale tan mal? Es complicado saberlo a ciencia cierta pero parece evidente que estamos haciendo overfitting. El sistema se comporta muy bien en el entrenamiento y muy mal en el conjunto de test. La razón puede ser que algunas de las variables tienen dependencia entre ellas. Si la dependencia no es lineal al probar la solución polinomial de grado 2 estaríamos aumentando el grado de tales dependencias. 
    <p>Por otro lado, vamos a probar con la codificación <b>Simple effect encoding</b> por si esta nueva configuración mejora el resultado.</p>
</div>

In [129]:
#Probemos ahora con otro tipo de encoding. Simple effect encoding.
for clave in np.array(cochesdf.keys()):
    if '_' in clave:
        cochesdf[clave] = cochesdf[clave]-0.25

In [131]:
#Comprobamos que se ha hecho bien.
print(cochesdf.head())

   doornumber  wheelbase  carlength  carwidth  carheight  curbweight  \
0           2       88.6      168.8      64.1       48.8        2548   
1           2       88.6      168.8      64.1       48.8        2548   
2           2       94.5      171.2      65.5       52.4        2823   
3           4       99.8      176.6      66.2       54.3        2337   
4           4       99.4      176.6      66.4       54.3        2824   

   enginesize  boreratio  stroke  compressionratio  ...  \
0         130       3.47    2.68               9.0  ...   
1         130       3.47    2.68               9.0  ...   
2         152       2.68    3.47               9.0  ...   
3         109       3.19    3.40              10.0  ...   
4         136       3.19    3.40               8.0  ...   

   cylindernumber_twelve  cylindernumber_two  fuelsystem_1bbl  \
0                  -0.25               -0.25            -0.25   
1                  -0.25               -0.25            -0.25   
2                

In [133]:
#Ahora volvemos a hacer la regresión polinómica. Esta vez el resultado es muy distinto.
#X = cochesdf.drop(['price'],axis=1)
#X = np.array(X)
X = cochesdf.drop(['price'],axis=1)
y = cochesdf['price']

from sklearn.preprocessing import PolynomialFeatures
poly_features = PolynomialFeatures(degree=2,include_bias=False)
X_poly = poly_features.fit_transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size = 0.2, random_state = 5)

from sklearn import metrics
from sklearn.metrics import mean_squared_error as MSE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)

print('Raiz del error cuadrático medio: ', MSE(y_test,y_pred,squared=False))
from sklearn.metrics import r2_score
print('R cuadrado: ', r2_score(y_test, y_pred))
print('R cuadrado entrenamiento: ', r2_score(y_train, lm.predict(X_train)))

Raiz del error cuadrático medio:  13668628.98441804
R cuadrado:  -2734867.4766733833
R cuadrado entrenamiento:  0.9992737484854026


<div style="border-style:groove;border-width:thin;padding:10px">
Comprobado que el Simple effect coding no soluciona el problema lo más proable es que las entradas tengan relación cuadrática entre ellas. Al hacer un polinomial estamos subiendo mucho el grado. Si por ejemplo la entrada w es w=x² cuando elevamos al cuadrado a w realmente estamos elevando a la 4 a x. Por eso estamos viendo overfitting. Para hacer un polinomial habría que buscar las variables de entrada que tengan menos correlación entre ellas.
    <p>Vamos a tratar de escoger solo entradas independientes. Que no tengan relación entre ellas.</p>
</div>

In [158]:
coches_mod = pd.DataFrame()
#coches_mod['wheelbase'] = cochesdf['wheelbase']
#coches_mod['carlength'] = cochesdf['carlength']
coches_mod['carwidth'] = cochesdf['carwidth']
#coches_mod['curbweight'] = cochesdf['curbweight']
coches_mod['enginesize'] = cochesdf['enginesize']
coches_mod['boreratio'] = cochesdf['boreratio']
#coches_mod['horsepower'] = cochesdf['horsepower']
#coches_mod['citympg'] = cochesdf['citympg']
coches_mod['highwaympg'] = cochesdf['highwaympg']
coches_mod['price'] = cochesdf['price']
coches_mod['drivewheel_rwd'] = cochesdf['drivewheel_rwd']
coches_mod['cylindernumber_four'] = cochesdf['cylindernumber_four']

print(coches_mod.corr()['price'])
print(coches_mod.corr()['cylindernumber_four'])



carwidth               0.759325
enginesize             0.874145
boreratio              0.553173
highwaympg            -0.697599
price                  1.000000
drivewheel_rwd         0.638957
cylindernumber_four   -0.697762
Name: price, dtype: float64
carwidth              -0.523135
enginesize            -0.631431
boreratio             -0.164076
highwaympg             0.547326
price                 -0.697762
drivewheel_rwd        -0.434461
cylindernumber_four    1.000000
Name: cylindernumber_four, dtype: float64


<div style="border-style:groove;border-width:thin;padding:10px">
Para seleccionar las columnas he quitado las columnas con menos correlación con el precio. Después he eliminado las que tienen una alta correlación entre ellas. Para ello he mirado la correlación entre ellas una por una por orden descendente de correlación con el precio eliminando las que más correlación tienen.
</div>

In [164]:
X = coches_mod.drop(['price'],axis=1)
y = cochesdf['price']

from sklearn.preprocessing import PolynomialFeatures
poly_features = PolynomialFeatures(degree=2,include_bias=False)
X_poly = poly_features.fit_transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size = 0.2, random_state = 0)

from sklearn import metrics
from sklearn.metrics import mean_squared_error as MSE
from sklearn.linear_model import LinearRegression

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)

print('Raiz del error cuadrático medio: ', MSE(y_test,y_pred,squared=False))
from sklearn.metrics import r2_score
print('R cuadrado: ', r2_score(y_test, y_pred))
print('R cuadrado entrenamiento: ', r2_score(y_train, lm.predict(X_train)))

Raiz del error cuadrático medio:  3351.2609331822073
R cuadrado:  0.8549279493354642
R cuadrado entrenamiento:  0.927669158863412
